[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-insuranceclaims.ipynb)

# Full Project: Insurance Claims Severity Prediction

*AIBits Academy · Machine Learning End To End · Full Project*

188,318 anonymized claims, 130 features, and a heavily skewed target — where Gradient Boosting edges out Random Forest on log-transformed loss.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

def fetch(url, target, member=None):   # public source; a zip member is extracted and renamed to `target`
    if os.path.exists(target):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if member:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(member)
    open(target, 'wb').write(blob)
    print('downloaded', target)

fetch('https://raw.githubusercontent.com/fardil-b/Insurance-Claims-Severity-Prediction/main/dataset.zip', 'train.csv', 'train.csv')

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> An insurance company processes claims manually, which is slow and expensive. Predicting the likely **severity** (monetary loss) of a claim as soon as it's filed — the same challenge faced by any general or health insurer, including India's growing private insurance sector — lets the company route high-severity claims to senior adjusters immediately and fast-track low-severity ones, improving both cost and customer experience.

> **Dataset**
>
> **188,318 insurance claims, 130 anonymized features** (116 categorical: `cat1`–`cat116`; 14 continuous: `cont1`–`cont14`) plus the target `loss`. Features are anonymized for confidentiality, so this project leans entirely on statistical structure rather than domain labels. [Dataset source (zip) →](https://raw.githubusercontent.com/fardil-b/Insurance-Claims-Severity-Prediction/main/dataset.zip)

## Step 1 — The Target Is Heavily Skewed

In [ ]:
import pandas as pd
train = pd.read_csv('train.csv')
print(train['loss'].describe())
print("Skewness:", train['loss'].skew())

Mean (₹3,037) far exceeds median (₹2,116) — a classic positive-skew signature (see Descriptive Statistics), driven by a small number of catastrophic claims reaching over ₹120,000. Left untreated, a regression model trained directly on this target would be dominated by the handful of extreme claims and fit the typical claim poorly. The standard fix, applied here, is a **log transform**: `y = log1p(loss)`.

## Step 2 — High Cardinality and Near-Zero-Variance Features

In [ ]:
cat_cols = [c for c in train.columns if c.startswith('cat')]
card = train[cat_cols].nunique()
print(card.sort_values(ascending=False).head(3))
print("Median cardinality:", card.median())

The 116 categorical features range from simple binary flags to `cat116`'s 326 distinct levels — label encoding, not one-hot encoding, is used to avoid an unmanageable explosion of dummy columns. A **Variance Threshold** filter then removes features that barely vary at all and therefore carry little to no predictive signal:

Build the feature matrix the text describes: label-encode the 116 categorical columns (no one-hot explosion), keep the 14 continuous ones, and set `loss` aside as the target.

In [ ]:
from sklearn.preprocessing import LabelEncoder

y_raw = train['loss']
X = train.drop(columns=['id', 'loss'])
for c in cat_cols:
    X[c] = LabelEncoder().fit_transform(X[c])
print(X.shape)

In [ ]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.01)
vt.fit(X)
print(f"Dropped {(~vt.get_support()).sum()} of {X.shape[1]} features as near-constant")

## Step 3 — Model Comparison on Log-Transformed Loss

> **⚠ Computed on a 40,000-row sample**
>
> The full training set has 188,318 rows across 130 mostly-categorical features; to keep this notebook's runtime reasonable, the model-fitting step below runs on a random 40,000-row sample of the real dataset (not synthetic data) — the same technique used to keep exploratory iterations fast on genuinely large datasets before a final full-data training run.

Drop the near-constant features found above, split 80/20, and log-transform the training target.

In [ ]:
from sklearn.model_selection import train_test_split

kept_features = X.columns[vt.get_support()]
X_kept = X[kept_features]
X_train, X_test, y_train, y_test = train_test_split(X_kept, y_raw, test_size=0.2, random_state=42)
y_train_log, y_test_log = np.log1p(y_train), np.log1p(y_test)
actual_rupees = y_test.values

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

y_log = np.log1p(y_raw)
# ... fit on log-transformed target, predict, then np.expm1() back to ₹ scale ...
for name, model in [('RandomForest', RandomForestRegressor(n_estimators=80, max_depth=10, min_samples_leaf=20)),
                    ('GradientBoosting', GradientBoostingRegressor(n_estimators=80, max_depth=4, learning_rate=0.1))]:
    model.fit(X_train, y_train_log)
    pred_log = model.predict(X_test)
    pred_rupees = np.expm1(pred_log)
    print(f"{name:18s} MAE=₹{mean_absolute_error(actual_rupees,pred_rupees):.2f}  R²(log scale)={r2_score(y_test_log,pred_log):.4f}")

**Gradient Boosting wins** — a lower mean absolute error (₹1,211.52 vs ₹1,279.05, roughly ₹67 tighter per prediction on average) and higher R² on the log-transformed scale (0.524 vs 0.468). This mirrors a broader pattern seen across this course's ensemble methods (see Boosting): gradient boosting's sequential error-correction often edges out Random Forest's independent-tree averaging on structured tabular data with many weak, noisy features like this one.

## Step 4 — Which Anonymized Features Actually Matter

The feature-importance table comes from a Random Forest fitted on the same log-scale target.

In [ ]:
rf = RandomForestRegressor(n_estimators=80, max_depth=10, min_samples_leaf=20, random_state=0, n_jobs=-1).fit(X_train, y_train_log)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=kept_features).sort_values(ascending=False)
print(importances.head(5))

A single feature, **cat80**, accounts for nearly 45% of the Random Forest's total feature importance — dramatically more than any other single feature. Since the features are anonymized, the business meaning of cat80 isn't visible from the data alone, but its dominance is itself the actionable finding: whatever cat80 represents (likely a claim-type or policy-type category, based on where it sits in the feature list), it deserves priority attention in any manual review process or future data-collection effort, precisely the same "let feature importance redirect business priorities" lesson as the Big Mart Sales full project.

## Visualizing cat80's Dominance

cat80 alone is nearly 6× the size of the next-highest feature.

## Key Business Takeaways

- The loss target is heavily right-skewed (skew=3.79) — a log transform is essential before regression, the same fix used for house prices and hospital bills discussed on the Descriptive Statistics page.
- Gradient Boosting (MAE=₹1,211.52) outperforms Random Forest (MAE=₹1,279.05) on this dataset — roughly a 5% tighter average error, consistent with boosting's general edge on wide, noisy tabular data.
- A single anonymized feature (cat80) drives nearly half of the model's predictive power — a strong signal for where to focus both model interpretation effort and future data-quality investment.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Skew before and after the log

Store the skewness of `train["loss"]` in `skew_raw` and of `np.log1p(train["loss"])` in `skew_log`.

In [ ]:
skew_raw = skew_log = None   # TODO


In [ ]:
try:
    check("raw about 3.79", abs(skew_raw - 3.79) < 0.05)
    check("log is nearly symmetric", abs(skew_log) < 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
skew_raw = float(train["loss"].skew())
skew_log = float(np.log1p(train["loss"]).skew())

```

</details>

### Exercise 2 · Medium · High-cardinality categoricals

Store in `high_card` the list of categorical columns (`cat_cols`) that have **more than 50** distinct levels, sorted by name.

In [ ]:
high_card = None   # TODO


In [ ]:
try:
    ref = sorted(c for c in cat_cols if train[c].nunique() > 50)
    check("same list", high_card == ref)
    check("contains cat116", "cat116" in high_card)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
high_card = sorted(c for c in cat_cols if train[c].nunique() > 50)

```

</details>

### Exercise 3 · Stretch · Beat the median predictor

A model that always predicts the median training loss is the floor. Compute its MAE on `actual_rupees` in `mae_median`, the gradient-boosting model's MAE (the last `pred_rupees` from the lesson's loop) in `mae_gb`, and `gb_wins`.

In [ ]:
mae_median = mae_gb = gb_wins = None   # TODO


In [ ]:
try:
    check("median baseline is worse", gb_wins is True)
    check("values", mae_gb < mae_median and abs(mae_gb - 1211.5) < 250)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
mae_median = mean_absolute_error(actual_rupees, np.full(len(actual_rupees), y_train.median()))
mae_gb = mean_absolute_error(actual_rupees, pred_rupees)
gb_wins = bool(mae_gb < mae_median)

```

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Insurance Claims Severity Prediction**.*